# Ejercicio 3 — Pipeline de preparación para SSD MC1

Se utilizan **las dos características de la sesión anterior**, ya calculadas en `OBT_MC1_RN`. El objetivo es preparar las observaciones para un modelo que prediga fallas durante los próximos 30 días. Las consultas se ejecutan en Snowflake y solo se recupera una muestra pequeña.

El cuaderno no entrena un modelo ni modifica las tablas de Snowflake.

## Paso 1. Clasificación de las columnas

Todas las variables de entradas son numericas, dentro del dataset no existen las variables categoricas ademas del
target. La variable que queremos predecir es `TARGET_30D`, con valores 0 y 1.

No se incluyen variables categóricas ni otras variables listas para el modelo. `SSD_KEY` y `OBSERVATION_DATE` solo sirven para identificar y separar las observaciones. `LABEL_STATUS`, las fechas de falla y los días hasta la falla quedan fuera de las entradas del modelo, ya que podrían revelar el resultado que intentamos predecir.

### 1.1. Importar librerías y definir las dos variables

In [1]:
import os
import re
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, MissingIndicator
from snowflake_io import create_spark_session, read_snowflake_query

#? Características calculadas en el cuaderno anterior
smart = ["R_1", "N_1", "R_5", "N_5", "R_9", "N_9"]
ventanas = [7, 14, 30]
tipos = ["MEAN", "COUNT", "MEAN_SHIFT"]

NUMERICAS = [
    f"{campo}_{tipo}_{dias}D"
    for campo in smart
    for dias in ventanas
    for tipo in tipos
]

OBJETIVO = "TARGET_30D"

print("Total de características numéricas:", len(NUMERICAS))

print("Variables numéricas:", NUMERICAS)
print("Objetivo:", OBJETIVO)
print("Variables categóricas: ninguna")

Total de características numéricas: 54
Variables numéricas: ['R_1_MEAN_7D', 'R_1_COUNT_7D', 'R_1_MEAN_SHIFT_7D', 'R_1_MEAN_14D', 'R_1_COUNT_14D', 'R_1_MEAN_SHIFT_14D', 'R_1_MEAN_30D', 'R_1_COUNT_30D', 'R_1_MEAN_SHIFT_30D', 'N_1_MEAN_7D', 'N_1_COUNT_7D', 'N_1_MEAN_SHIFT_7D', 'N_1_MEAN_14D', 'N_1_COUNT_14D', 'N_1_MEAN_SHIFT_14D', 'N_1_MEAN_30D', 'N_1_COUNT_30D', 'N_1_MEAN_SHIFT_30D', 'R_5_MEAN_7D', 'R_5_COUNT_7D', 'R_5_MEAN_SHIFT_7D', 'R_5_MEAN_14D', 'R_5_COUNT_14D', 'R_5_MEAN_SHIFT_14D', 'R_5_MEAN_30D', 'R_5_COUNT_30D', 'R_5_MEAN_SHIFT_30D', 'N_5_MEAN_7D', 'N_5_COUNT_7D', 'N_5_MEAN_SHIFT_7D', 'N_5_MEAN_14D', 'N_5_COUNT_14D', 'N_5_MEAN_SHIFT_14D', 'N_5_MEAN_30D', 'N_5_COUNT_30D', 'N_5_MEAN_SHIFT_30D', 'R_9_MEAN_7D', 'R_9_COUNT_7D', 'R_9_MEAN_SHIFT_7D', 'R_9_MEAN_14D', 'R_9_COUNT_14D', 'R_9_MEAN_SHIFT_14D', 'R_9_MEAN_30D', 'R_9_COUNT_30D', 'R_9_MEAN_SHIFT_30D', 'N_9_MEAN_7D', 'N_9_COUNT_7D', 'N_9_MEAN_SHIFT_7D', 'N_9_MEAN_14D', 'N_9_COUNT_14D', 'N_9_MEAN_SHIFT_14D', 'N_9_MEAN_30D', 'N_9_C

### 1.2. Conectar con Snowflake

Se emplea el mismo módulo `snowflake_io` que en el cuaderno anterior. La base de datos se toma del entorno, por lo
que también funciona con la base compartida de la cuenta de prueba. Para la conexion en este documento vamos a
utilizar la `OBT_MC1_RN` que contiene la OBT final para la evaluacion de las metricas, sue scalamiento y limpieza.
Aunque la ingeniera de features ya se hizo para crear esta tabla, al entrenar se tendra que realizar un procesamiento
 mayor sobre las columnas registradas.

In [2]:
spark = create_spark_session("mc1-pipeline-preparacion")
database = os.environ["SNOWFLAKE_DATABASE"].strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*", database):
    raise ValueError("Nombre de base de datos no válido")

obt = f'"{database}"."S_CDATOS_PSET2_OBT"."OBT_MC1_RN"'

def consultar(sql, max_filas=200_000):
    # Limitar el resultado que se trae desde Snowflake.
    texto = sql.strip().rstrip(";")
    if not texto.upper().startswith(("SELECT", "WITH")) or ";" in texto:
        raise ValueError("Solo se permiten consultas de lectura")
    resultado = read_snowflake_query(
        spark,
        f"SELECT * FROM ({texto}) q LIMIT {max_filas + 1}",
        schema="S_CDATOS_PSET2_OBT"
    ).toPandas()
    if len(resultado) > max_filas:
        raise ValueError("La muestra supera el límite de filas")
    return resultado

print("Tabla de origen:", obt)
print("Versión de Spark:", spark.version)

26/10/26 23:40:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Tabla de origen: "WXVGCFK_QF18928_S_CDATOS_PSET2_SNOWFLAKE_SHARE_1792893424527"."S_CDATOS_PSET2_OBT"."OBT_MC1_RN"
Versión de Spark: 4.0.4


### 1.3. Obtener una muestra con las dos características

La OBT contiene las características de la sesión anterior, por lo que **no es necesario volver a calcularlas**. Para evitar descargar millones de observaciones, Snowflake selecciona aproximadamente el **0,1 %** de las filas mediante un muestreo aleatorio.

Solo se recuperan observaciones con `TARGET_30D` conocido. Las fechas de diciembre de 2018 cercanas al corte se reservan como separación temporal. Esta muestra conserva aproximadamente el desbalance original, pero no sustituye a la población completa para entrenar un modelo definitivo.

In [3]:
#? Seleccionar hasta 50 000 observaciones con las 54 características.
#? Mantener las particiones cronológicas y los embargos de julio y octubre.

columnas_sql = ",\n            ".join(NUMERICAS)

datos = consultar(f"""
    WITH particiones AS (
        SELECT
            SSD_KEY,
            OBSERVATION_DATE,
            {columnas_sql},
            TARGET_30D,
            CASE
                WHEN OBSERVATION_DATE < '2019-07-01'::DATE
                    THEN 'ENTRENAMIENTO'
                WHEN OBSERVATION_DATE BETWEEN '2019-08-01' AND '2019-09-30'
                    THEN 'VALIDACION'
                WHEN OBSERVATION_DATE BETWEEN '2019-11-01' AND '2019-11-30'
                    THEN 'PRUEBA'
            END AS PARTICION
        FROM {obt}
        WHERE TARGET_30D IN (0, 1)
          AND (
              OBSERVATION_DATE < '2019-07-01'::DATE
              OR OBSERVATION_DATE BETWEEN '2019-08-01' AND '2019-09-30'
              OR OBSERVATION_DATE BETWEEN '2019-11-01' AND '2019-11-30'
          )
    ),
    muestra AS (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY PARTICION
                   ORDER BY RANDOM()
               ) AS FILA
        FROM particiones
    )
    SELECT
        SSD_KEY,
        OBSERVATION_DATE,
        {columnas_sql},
        TARGET_30D,
        PARTICION
    FROM muestra
    WHERE (PARTICION = 'ENTRENAMIENTO' AND FILA <= 30000)
       OR (PARTICION = 'VALIDACION' AND FILA <= 10000)
       OR (PARTICION = 'PRUEBA' AND FILA <= 10000)
""", max_filas=50000)

#? Convertir las columnas y comprobar la muestra.
datos["OBSERVATION_DATE"] = pd.to_datetime(datos["OBSERVATION_DATE"])
datos[NUMERICAS] = datos[NUMERICAS].apply(pd.to_numeric, errors="coerce")

print("Observaciones recuperadas:", len(datos))
print("Características numéricas:", len(NUMERICAS))

datos

26/10/26 23:40:35 WARN NoOpFileCacheManager: Cache is not available. Caching will be disabled. Set the HOME, SF_TEMPORARY_CREDENTIAL_CACHE_DIR, or net.snowflake.jdbc.temporaryCredentialCacheDir to enable caching.
26/10/26 23:40:36 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,035 ms, timeout: 1,000 ms
26/10/26 23:40:36 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,036 ms, timeout: 1,000 ms
26/10/26 23:40:36 ERROR RestRequest: Returning null response. Cause: java.net.UnknownHostException: metadata.google.internal, request: GET http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/email HTTP/1.1
26/10/26 23:40:36 ERROR RestRequest: Returning null response. Cause: java.net.UnknownHostException: metadata.google.internal, request: GET http://metadata.google.internal HTTP/1.1
26/10/26 23:40:37 ERROR RestRequest: Stop retrying since elapsed tim

Observaciones recuperadas: 50000
Características numéricas: 54


,SSD_KEY,OBSERVATION_DATE,R_1_MEAN_7D,R_1_COUNT_7D,R_1_MEAN_SHIFT_7D,R_1_MEAN_14D,R_1_COUNT_14D,R_1_MEAN_SHIFT_14D,R_1_MEAN_30D,R_1_COUNT_30D,...,N_9_COUNT_7D,N_9_MEAN_SHIFT_7D,N_9_MEAN_14D,N_9_COUNT_14D,N_9_MEAN_SHIFT_14D,N_9_MEAN_30D,N_9_COUNT_30D,N_9_MEAN_SHIFT_30D,TARGET_30D,PARTICION
0,7c3447ed5566c51ebf869fdca0c9840443f5dfca0aa122...,2018-10-07,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
1,be9c557194bac99600c65fecd64e3950303952214b928c...,2018-03-14,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
2,512e77ea66605c216911d44e82f99c60089efdcc9f5adf...,2018-04-07,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
3,1f3f5a2a761febb40e5cf21d44f2b241990d95adde1ca5...,2018-12-14,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
4,f5a55f1c49b034956ac56d60641566370c1fae63223fff...,2019-03-21,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,10cbd5804010b98b16e7b13ed5996074832124267c0a51...,2018-09-14,0.0,6.0,0.0,0.0,13.0,0.0,0.0,29.0,...,6.0,0.0,100.0,13.0,0.0,100.0,29.0,0.0,0,ENTRENAMIENTO
49996,4d58499322fc47c26123e64901bbdf946aee0844d68c6c...,2018-01-23,0.0,7.0,0.0,0.0,14.0,0.0,0.0,21.0,...,7.0,0.0,100.0,14.0,0.0,100.0,21.0,NaN,0,ENTRENAMIENTO
49997,0a3b78d931369b70fdf0de18c61e8148bf519d246a2344...,2019-03-03,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
49998,8ca6af732cb5d8d23ccdefde69450b90abc3eb991c2e1b...,2018-08-11,0.0,7.0,0.0,0.0,14.0,0.0,0.0,29.0,...,7.0,0.0,100.0,14.0,0.0,100.0,29.0,0.0,0,ENTRENAMIENTO


#### 1.3.1 Variacion de seleccion de datos con incremento en la seccion de casos positivos en training
Para esta subseccion, la query ejecutada permite seleccionar de manera aleatoria un mayor porcentaje de casos positivos en el conjunto de entrenamiento, con el objetivo de evaluar el impacto de este cambio en la preparación de los datos y en la distribución de las clases. La selección se realiza manteniendo las particiones cronológicas y los embargos establecidos previamente.

In [9]:
#? Seleccionar hasta 50 000 observaciones respetando las particiones
#? cronológicas y los embargos definidos para el proyecto.
#?
#? Entrenamiento: máximo 5 000 positivos y 25 000 negativos.
#? Validación: máximo 10 000 observaciones, sin balancear clases.
#? Prueba: máximo 10 000 observaciones, sin balancear clases.

#? Construir la lista SQL de las 54 características numéricas.
#? El prefijo "o." identifica las columnas de la tabla OBT
#? cuando se realiza el JOIN final.
columnas_sql = ",\n        ".join(f"o.{col}" for col in NUMERICAS)

datos_higher_pos_density = consultar(
    f"""
    -- CTE 1: CANDIDATOS
    -- Identificar las observaciones elegibles y asignarlas
    -- a entrenamiento, validación o prueba según su fecha.
    WITH candidatos AS (
        SELECT
            SSD_KEY,            -- Identificador de la unidad SSD.
            OBSERVATION_DATE,   -- Fecha de la medición SMART.
            TARGET_30D,        -- 1: falla futura; 0: sin falla futura.
            -- Asignar una partición según los períodos definidos.
            CASE
                -- Entrenamiento: hasta el 30 de junio de 2019.
                WHEN OBSERVATION_DATE < '2019-07-01'::DATE
                    THEN 'ENTRENAMIENTO'

                -- Validación: agosto y septiembre de 2019.
                -- Julio queda excluido como período de embargo.
                WHEN OBSERVATION_DATE
                     BETWEEN '2019-08-01' AND '2019-09-30'
                    THEN 'VALIDACION'

                -- Prueba: noviembre de 2019.
                -- Octubre queda excluido como segundo embargo.
                WHEN OBSERVATION_DATE
                     BETWEEN '2019-11-01' AND '2019-11-30'
                    THEN 'PRUEBA'

                -- Las demás fechas reciben NULL y se descartan.
            END AS PARTICION

        -- Leer las observaciones de la tabla OBT existente.
        FROM {obt}

        WHERE
            -- Excluir etiquetas censuradas o desconocidas.
            TARGET_30D IN (0, 1)

            AND (
                -- Conservar el período de entrenamiento.
                OBSERVATION_DATE < '2019-07-01'::DATE

                -- Conservar el período de validación.
                OR OBSERVATION_DATE
                   BETWEEN '2019-08-01' AND '2019-09-30'

                -- Conservar el período de prueba.
                OR OBSERVATION_DATE
                   BETWEEN '2019-11-01' AND '2019-11-30'
            )
    ),

    -- CTE 2: REDUCIDOS
    -- Disminuir la cantidad de candidatos antes de numerarlos.
    -- IMPORTANTE: los positivos nunca se descartan aquí.
    reducidos AS (
        SELECT *
        FROM candidatos

        -- Excluir fechas que no pertenezcan a una partición.
        WHERE PARTICION IS NOT NULL

          AND (
              -- Conservar TODOS los positivos elegibles.
              TARGET_30D = 1

              -- Conservar TODOS los candidatos de validación
              -- y prueba, independientemente de su clase.
              OR PARTICION IN ('VALIDACION', 'PRUEBA')

              -- Para los negativos de entrenamiento:
              -- conservar aproximadamente el 5 % usando
              -- un hash determinista del SSD y de la fecha.
              --
              -- MOD(..., 100) < 5 selecciona 5 residuos
              -- de un total de 100 posibles.
              --
              -- ABS evita residuos negativos.
              OR MOD(
                  ABS(HASH(SSD_KEY, OBSERVATION_DATE)),
                  100
              ) < 5
          )
    ),

    -- CTE 3: NUMERADOS
    -- Asignar números de fila para aplicar cuotas.
    numerados AS (
        SELECT
            *,

            -- Número de fila dentro de cada combinación
            -- de partición y clase.
            --
            -- Ejemplo: los positivos de entrenamiento
            -- se enumeran independientemente de los negativos.
            ROW_NUMBER() OVER (
                PARTITION BY PARTICION, TARGET_30D
                ORDER BY HASH(SSD_KEY, OBSERVATION_DATE)
            ) AS FILA_CLASE,

            -- Número de fila dentro de cada partición,
            -- mezclando ambas clases según el hash.
            --
            -- Se utiliza para seleccionar 10 000 registros
            -- de validación y 10 000 de prueba.
            ROW_NUMBER() OVER (
                PARTITION BY PARTICION
                ORDER BY HASH(SSD_KEY, OBSERVATION_DATE)
            ) AS FILA_PARTICION

        FROM reducidos
    ),

    -- CTE 4: SELECCION
    -- Aplicar las cuotas de observaciones a cada conjunto.
    seleccion AS (
        SELECT
            SSD_KEY,
            OBSERVATION_DATE,
            TARGET_30D,
            PARTICION

        FROM numerados

        WHERE (
            -- Cuotas específicas de entrenamiento.
            PARTICION = 'ENTRENAMIENTO'

            AND (
                -- Máximo 5 000 observaciones positivas.
                (TARGET_30D = 1 AND FILA_CLASE <= 5000)

                OR

                -- Máximo 25 000 observaciones negativas.
                (TARGET_30D = 0 AND FILA_CLASE <= 25000)
            )
        )

        OR (
            -- Validación y prueba mantienen un muestreo
            -- sin cuotas específicas para sus clases.
            PARTICION IN ('VALIDACION', 'PRUEBA')

            -- Seleccionar hasta 10 000 filas por partición.
            AND FILA_PARTICION <= 10000
        )
    )

    -- CONSULTA FINAL
    -- Recuperar las 54 características únicamente para
    -- las observaciones seleccionadas anteriormente.
    SELECT
        o.SSD_KEY,
        o.OBSERVATION_DATE,

        -- Insertar las 54 columnas generadas con NUMERICAS.
        {columnas_sql},

        -- Conservar la etiqueta y la partición para
        -- separar X, y y los tres conjuntos.
        s.TARGET_30D,
        s.PARTICION

    FROM seleccion s

    -- Recuperar las características de la OBT original.
    JOIN {obt} o
      ON o.SSD_KEY = s.SSD_KEY
     AND o.OBSERVATION_DATE = s.OBSERVATION_DATE

""", max_filas=50000)

#? Convertir las fechas y características a sus tipos en pandas.
datos_higher_pos_density["OBSERVATION_DATE"] = pd.to_datetime(datos_higher_pos_density["OBSERVATION_DATE"])
datos_higher_pos_density[NUMERICAS] = datos_higher_pos_density[NUMERICAS].apply(
    pd.to_numeric, errors="coerce"
)

#? Comprobar la distribución de clases resultante.
datos_higher_pos_density.groupby(
    ["PARTICION", "TARGET_30D"]
).size().unstack(fill_value=0)

TARGET_30D,0,1
PARTICION,,
ENTRENAMIENTO,25000,5000
PRUEBA,9959,41
VALIDACION,9936,64


In [11]:
datos_higher_pos_density.loc[
    (datos_higher_pos_density["PARTICION"] == "ENTRENAMIENTO")
    & (datos_higher_pos_density["TARGET_30D"] == 1),
    "SSD_KEY"
].nunique()

3246

### 1.3 Validar los datos obtenidos por la muestra

In [4]:
#? Verificar la cantidad de observaciones y clases por partición.

resumen = (
    datos.groupby(["PARTICION", "TARGET_30D"])
    .size()
    .unstack(fill_value=0)
)

resumen["TOTAL"] = resumen.sum(axis=1)
resumen

TARGET_30D,0,1,TOTAL
PARTICION,,,
ENTRENAMIENTO,29952,48,30000
PRUEBA,9965,35,10000
VALIDACION,9933,67,10000


In [5]:
# La selección y el filtrado se realizan dentro de Snowflake.

datos["OBSERVATION_DATE"] = pd.to_datetime(datos["OBSERVATION_DATE"])
datos[NUMERICAS] = datos[NUMERICAS].apply(pd.to_numeric, errors="coerce")
datos[OBJETIVO] = pd.to_numeric(datos[OBJETIVO], errors="raise").astype(int)

print("Observaciones de la muestra:", len(datos))
print("Valores del objetivo:")
print(datos[OBJETIVO].value_counts().sort_index())
datos.head(6)

Observaciones de la muestra: 50000
Valores del objetivo:
TARGET_30D
0    49850
1      150
Name: count, dtype: int64


,SSD_KEY,OBSERVATION_DATE,R_1_MEAN_7D,R_1_COUNT_7D,R_1_MEAN_SHIFT_7D,R_1_MEAN_14D,R_1_COUNT_14D,R_1_MEAN_SHIFT_14D,R_1_MEAN_30D,R_1_COUNT_30D,...,N_9_COUNT_7D,N_9_MEAN_SHIFT_7D,N_9_MEAN_14D,N_9_COUNT_14D,N_9_MEAN_SHIFT_14D,N_9_MEAN_30D,N_9_COUNT_30D,N_9_MEAN_SHIFT_30D,TARGET_30D,PARTICION
0,7c3447ed5566c51ebf869fdca0c9840443f5dfca0aa122...,2018-10-07,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
1,be9c557194bac99600c65fecd64e3950303952214b928c...,2018-03-14,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
2,512e77ea66605c216911d44e82f99c60089efdcc9f5adf...,2018-04-07,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
3,1f3f5a2a761febb40e5cf21d44f2b241990d95adde1ca5...,2018-12-14,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
4,f5a55f1c49b034956ac56d60641566370c1fae63223fff...,2019-03-21,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
5,946c5434078ed096ca93c131c91f7041e5fc723b82b849...,2018-02-28,0.0,7.0,0.0,0.0,14.0,0.0,0.0,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO


## Paso 2. División temporal

Dado que el modelo busca anticipar fallas futuras en las unidades SSD, se utiliza una división cronológica de los registros, respetando los períodos de entrenamiento, validación y prueba definidos para el proyecto.

El conjunto de **entrenamiento** comprende las observaciones registradas hasta el 30 de junio de 2019. Posteriormente, se establece julio como un período de separación de 30 días. El conjunto de **validación** utiliza las observaciones de agosto y septiembre de 2019, mientras que octubre constituye un segundo período de separación. Finalmente, el conjunto de **prueba** comprende las observaciones de noviembre de 2019.

Estos períodos de separación se establecen porque la variable `TARGET_30D` considera la posibilidad de una falla
durante los 30 días posteriores a cada observación. De esta manera, se busca evitar que las etiquetas utilizadas
durante el entrenamiento y la validación dependan de eventos correspondientes al conjunto previo.

Para realizar el ejercicio, se seleccionó una muestra máxima de **50.000 observaciones**, distribuida entre 30.000 registros de entrenamiento, 10.000 de validación y 10.000 de prueba. Esta selección mantiene las fechas establecidas y permite ejecutar el proceso de preparación sin cargar la totalidad de los registros en el entorno local.

La separación se realiza por fechas, por lo que una misma unidad SSD puede aparecer en diferentes conjuntos. Esta condición deberá considerarse durante la evaluación posterior del modelo.

### 2.1. Separar las observaciones por fecha

In [12]:
#? Separar las observaciones según las particiones definidas en Snowflake.
entrenamiento = datos.loc[datos["PARTICION"] == "ENTRENAMIENTO"].copy()
validacion = datos.loc[datos["PARTICION"] == "VALIDACION"].copy()
prueba = datos.loc[datos["PARTICION"] == "PRUEBA"].copy()

#? Verificar que los tres conjuntos contienen observaciones.
if entrenamiento.empty or validacion.empty or prueba.empty:
    raise ValueError("No hay observaciones suficientes en alguno de los períodos")

#? Mostrar los rangos y cantidades de cada conjunto.
for nombre, conjunto in [
    ("Entrenamiento", entrenamiento),
    ("Validación", validacion),
    ("Prueba", prueba),
]:
    print(f"{nombre}:")
    print("  Primera observación:", conjunto["OBSERVATION_DATE"].min())
    print("  Última observación:", conjunto["OBSERVATION_DATE"].max())
    print("  Total de registros:", len(conjunto))

Entrenamiento:
  Primera observación: 2018-01-03 00:00:00
  Última observación: 2019-06-30 00:00:00
  Total de registros: 30000
Validación:
  Primera observación: 2019-08-01 00:00:00
  Última observación: 2019-09-30 00:00:00
  Total de registros: 10000
Prueba:
  Primera observación: 2019-11-01 00:00:00
  Última observación: 2019-11-30 00:00:00
  Total de registros: 10000


### 2.2. Separar las entradas y el objetivo

Solo se entregan las dos características numéricas al modelo. Las fechas, los identificadores y las etiquetas descriptivas no forman parte de `X`.

In [13]:
X_ent = entrenamiento[NUMERICAS]
y_ent = entrenamiento[OBJETIVO]

X_val = validacion[NUMERICAS]
y_val = validacion[OBJETIVO]

X_prueba = prueba[NUMERICAS]
y_prueba = prueba[OBJETIVO]

### 2.3. Revisar valores faltantes

Algunas ventanas de cambio de media pueden no tener suficientes observaciones anteriores. Estos valores se completarán utilizando únicamente las medianas del entrenamiento.

In [14]:
nulos = pd.DataFrame({
    "ENTRENAMIENTO": X_ent.isna().sum(),
    "VALIDACIÓN": X_val.isna().sum(),
    "PRUEBA": X_prueba.isna().sum(),
})

if X_ent.notna().sum().min() == 0:
    raise ValueError("Una de las variables no tiene valores disponibles en entrenamiento")

nulos

,ENTRENAMIENTO,VALIDACIÓN,PRUEBA
R_1_MEAN_7D,0,0,0
R_1_COUNT_7D,0,0,0
R_1_MEAN_SHIFT_7D,596,88,118
R_1_MEAN_14D,0,0,0
R_1_COUNT_14D,0,0,0
R_1_MEAN_SHIFT_14D,1099,159,169
R_1_MEAN_30D,0,0,0
R_1_COUNT_30D,0,0,0
R_1_MEAN_SHIFT_30D,2167,205,203
N_1_MEAN_7D,0,0,0


## Paso 3. Construcción y ejecución del pipeline

Para cada una de las dos columnas numéricas se aplica la misma secuencia. Primero se completan los valores faltantes con la mediana del entrenamiento y después se ajustan las escalas numéricas.

**La preparación se ajusta solo con entrenamiento.** En prueba se reutilizan los mismos valores de referencia, sin volver a calcularlos.

### 3.1. Definir la preparación de variables numéricas

Para la preparación de los datos se decidió aplicar una **normalización mediante `StandardScaler`**, con el objetivo de establecer una escala comparable entre las características seleccionadas. Esta decisión resulta relevante debido a que `R_5_MEAN_SHIFT_7D` representa cambios en el promedio de sectores reasignados, mientras que `R_9_MEAN_30D` representa el promedio de las horas de funcionamiento acumuladas. Dado que ambas características presentan magnitudes diferentes, el escalado permite reducir estas diferencias y facilitar su utilización en modelos que pueden verse afectados por la escala de las variables. Los parámetros de esta transformación se calculan únicamente con el conjunto de entrenamiento y posteriormente se aplican a los conjuntos de validación y prueba.

En cuanto a los valores ausentes, se identificaron **592 registros incompletos durante el entrenamiento, 111 durante la validación y 133 durante la prueba** para la característica `R_5_MEAN_SHIFT_7D`. Estos valores pueden aparecer cuando no existen suficientes mediciones disponibles para calcular la diferencia entre los promedios de dos períodos consecutivos. Por esta razón, se decidió aplicar una **imputación mediante la mediana**, exclusivamente como parte de una prueba del proceso de preparación. Aunque esta operación permite conservar las observaciones sin eliminar registros, también introduce valores estimados que no necesariamente representan el comportamiento real de las unidades SSD. En consecuencia, su aplicación no implica que esta sea la estrategia definitiva para el modelo.

Finalmente, se incorporó un **indicador de valores ausentes** mediante `add_indicator=True`, que permite identificar las observaciones cuyo valor original fue reemplazado durante la imputación. De esta manera, el proceso conserva información sobre la ausencia de mediciones y permite distinguir entre los valores calculados originalmente y aquellos completados mediante la mediana. La combinación de estas tres operaciones se utiliza para comprobar el funcionamiento del proceso de preparación y obtener conjuntos de datos compatibles con posteriores etapas de entrenamiento. No obstante, la selección definitiva de estas transformaciones corresponderá a la evaluación del modelo que se realice en las siguientes etapas del proyecto.

In [15]:
#? Imputar y escalar únicamente las dos características numéricas.
#? Conservar el indicador de valores ausentes sin escalado.
preparacion = ColumnTransformer([
    ("numericas", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ]), NUMERICAS),

    ("indicador_nulos", MissingIndicator(features="missing-only"), NUMERICAS),
])

preparacion

,transformers,"[('numericas', ...), ('indicador_nulos', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### 3.2. Preparar los datos de entrenamiento

In [16]:
#? Ajustar la imputación y el escalado únicamente con entrenamiento.
X_ent_listo = preparacion.fit_transform(X_ent)

#? Aplicar las mismas transformaciones sin volver a ajustarlas.
X_val_listo = preparacion.transform(X_val)
X_prueba_listo = preparacion.transform(X_prueba)

#? Verificar las dimensiones y las columnas resultantes.
print("Entrenamiento:", X_ent_listo.shape)
print("Validación:", X_val_listo.shape)
print("Prueba:", X_prueba_listo.shape)

print("Columnas:", list(preparacion.get_feature_names_out()))

Entrenamiento: (30000, 72)
Validación: (10000, 72)
Prueba: (10000, 72)
Columnas: ['numericas__R_1_MEAN_7D', 'numericas__R_1_COUNT_7D', 'numericas__R_1_MEAN_SHIFT_7D', 'numericas__R_1_MEAN_14D', 'numericas__R_1_COUNT_14D', 'numericas__R_1_MEAN_SHIFT_14D', 'numericas__R_1_MEAN_30D', 'numericas__R_1_COUNT_30D', 'numericas__R_1_MEAN_SHIFT_30D', 'numericas__N_1_MEAN_7D', 'numericas__N_1_COUNT_7D', 'numericas__N_1_MEAN_SHIFT_7D', 'numericas__N_1_MEAN_14D', 'numericas__N_1_COUNT_14D', 'numericas__N_1_MEAN_SHIFT_14D', 'numericas__N_1_MEAN_30D', 'numericas__N_1_COUNT_30D', 'numericas__N_1_MEAN_SHIFT_30D', 'numericas__R_5_MEAN_7D', 'numericas__R_5_COUNT_7D', 'numericas__R_5_MEAN_SHIFT_7D', 'numericas__R_5_MEAN_14D', 'numericas__R_5_COUNT_14D', 'numericas__R_5_MEAN_SHIFT_14D', 'numericas__R_5_MEAN_30D', 'numericas__R_5_COUNT_30D', 'numericas__R_5_MEAN_SHIFT_30D', 'numericas__N_5_MEAN_7D', 'numericas__N_5_COUNT_7D', 'numericas__N_5_MEAN_SHIFT_7D', 'numericas__N_5_MEAN_14D', 'numericas__N_5_COUNT_1

### 3.3. Verificar las dimensiones y columnas resultantes

Una vez aplicada la preparación, se verifican las dimensiones de los conjuntos de entrenamiento, validación y prueba, junto con los nombres de las columnas generadas.

El proceso conserva las dos características numéricas originales, aunque sus valores han sido transformados mediante la imputación y el escalado. Además, se incorpora una tercera columna que permite identificar las observaciones que originalmente no tenían un valor disponible para `R_5_MEAN_SHIFT_7D`.

Esta verificación permite comprobar que los tres conjuntos mantienen las mismas columnas y que las transformaciones se aplicaron de manera consistente, utilizando únicamente los datos de entrenamiento para ajustar el proceso.

In [17]:
#? Obtener los nombres de las columnas generadas.
columnas_finales = list(preparacion.get_feature_names_out())

#? Verificar las dimensiones de los tres conjuntos.
print("Entrenamiento:", X_ent_listo.shape)
print("Validación:", X_val_listo.shape)
print("Prueba:", X_prueba_listo.shape)

#? Mostrar los nombres de las columnas después de la preparación.
print("\nColumnas producidas:")
for columna in columnas_finales:
    print("-", columna)

#? Verificar las clases presentes en cada conjunto.
print("\nClases del objetivo:")
print("Entrenamiento:", sorted(y_ent.unique().tolist()))
print("Validación:", sorted(y_val.unique().tolist()))
print("Prueba:", sorted(y_prueba.unique().tolist()))

Entrenamiento: (30000, 72)
Validación: (10000, 72)
Prueba: (10000, 72)

Columnas producidas:
- numericas__R_1_MEAN_7D
- numericas__R_1_COUNT_7D
- numericas__R_1_MEAN_SHIFT_7D
- numericas__R_1_MEAN_14D
- numericas__R_1_COUNT_14D
- numericas__R_1_MEAN_SHIFT_14D
- numericas__R_1_MEAN_30D
- numericas__R_1_COUNT_30D
- numericas__R_1_MEAN_SHIFT_30D
- numericas__N_1_MEAN_7D
- numericas__N_1_COUNT_7D
- numericas__N_1_MEAN_SHIFT_7D
- numericas__N_1_MEAN_14D
- numericas__N_1_COUNT_14D
- numericas__N_1_MEAN_SHIFT_14D
- numericas__N_1_MEAN_30D
- numericas__N_1_COUNT_30D
- numericas__N_1_MEAN_SHIFT_30D
- numericas__R_5_MEAN_7D
- numericas__R_5_COUNT_7D
- numericas__R_5_MEAN_SHIFT_7D
- numericas__R_5_MEAN_14D
- numericas__R_5_COUNT_14D
- numericas__R_5_MEAN_SHIFT_14D
- numericas__R_5_MEAN_30D
- numericas__R_5_COUNT_30D
- numericas__R_5_MEAN_SHIFT_30D
- numericas__N_5_MEAN_7D
- numericas__N_5_COUNT_7D
- numericas__N_5_MEAN_SHIFT_7D
- numericas__N_5_MEAN_14D
- numericas__N_5_COUNT_14D
- numericas__N_5

### 3.5. Consultar una muestra de los datos preparados

Esta tabla conserva la identificación y la etiqueta **solo para revisar los resultados**. Esas columnas no entraron en la preparación. El resultado se deja como DataFrame para poder abrirlo con Data Wrangler.

In [18]:
prueba_preparada = pd.DataFrame(
    X_prueba_listo, columns=columnas_finales, index=X_prueba.index
)
prueba_preparada.insert(0, "SSD_KEY", prueba["SSD_KEY"])
prueba_preparada.insert(1, "OBSERVATION_DATE", prueba["OBSERVATION_DATE"])
prueba_preparada["TARGET_30D"] = y_prueba

# Hasta 50 filas por clase, sin alterar los datos preparados.
muestra_preparada = (
    prueba_preparada.groupby("TARGET_30D", group_keys=False)
    .head(2500)
    .sort_values(["TARGET_30D", "OBSERVATION_DATE"])
    .reset_index(drop=True)
)

muestra_preparada

,SSD_KEY,OBSERVATION_DATE,numericas__R_1_MEAN_7D,numericas__R_1_COUNT_7D,numericas__R_1_MEAN_SHIFT_7D,numericas__R_1_MEAN_14D,numericas__R_1_COUNT_14D,numericas__R_1_MEAN_SHIFT_14D,numericas__R_1_MEAN_30D,numericas__R_1_COUNT_30D,...,indicador_nulos__missingindicator_N_5_MEAN_SHIFT_7D,indicador_nulos__missingindicator_N_5_MEAN_SHIFT_14D,indicador_nulos__missingindicator_N_5_MEAN_SHIFT_30D,indicador_nulos__missingindicator_R_9_MEAN_SHIFT_7D,indicador_nulos__missingindicator_R_9_MEAN_SHIFT_14D,indicador_nulos__missingindicator_R_9_MEAN_SHIFT_30D,indicador_nulos__missingindicator_N_9_MEAN_SHIFT_7D,indicador_nulos__missingindicator_N_9_MEAN_SHIFT_14D,indicador_nulos__missingindicator_N_9_MEAN_SHIFT_30D,TARGET_30D
0,3034f52bcd06c7175a38451b0246f2f4eab1ebf43cf47c...,2019-11-01,-0.007296,0.265125,0.004532,-0.007301,0.330709,-0.003967,-0.007298,0.393512,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1469d00d3034f661607f2f1fb636af8250d4845d02de3e...,2019-11-01,-0.007296,0.265125,0.004532,-0.007301,-0.831372,-0.003967,-0.007298,-0.025981,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,65b3dcd98469848a22f743d6d52b665cc84a8d84c3f78b...,2019-11-01,-0.007296,0.265125,0.004532,-0.007301,0.330709,-0.003967,-0.007298,0.393512,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,37afada3cff40f8661c10a26c426f0679aaec9a0a4fe2f...,2019-11-01,-0.006989,0.265125,0.004532,-0.006994,-0.250332,-0.003967,-0.006991,0.183766,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,f59d9b28264914af2209f9528715cc47b69a400c4d3660...,2019-11-01,-0.007296,0.265125,0.004532,-0.007301,-0.250332,-0.003967,-0.007298,0.183766,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2530,8f2df84d2f7dacde52a44dcdd77049925ab8ded097fbd3...,2019-11-27,-0.004532,0.265125,0.004532,-0.004537,0.330709,-0.003967,-0.004534,-0.445473,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2531,50267dd2800ded1c3d88915f46b459b9ee5ecdb3f17a10...,2019-11-28,0.004222,0.265125,0.004532,0.004217,-0.250332,-0.003967,0.004220,-0.864965,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2532,5b43a0d10e01984cd3adab2bd256caa1422bddc1651281...,2019-11-30,2.008954,0.265125,0.004532,2.008949,0.330709,-0.003967,2.008955,-0.655219,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2533,cae6c74ce5e6c3caceb604ba33437b240b7d740a27bfbe...,2019-11-30,0.058893,0.265125,0.004532,0.058888,0.330709,-0.003967,0.058891,-0.445473,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
